# Module 07 — The Extended Euclidean Algorithm
### Mathematics of Cryptography · Python Companion Notebook

This notebook builds the extended Euclidean algorithm from the ground up — back-substitution traces, iterative implementation, modular inverses, and practical crypto applications.
Every example mirrors Tutorial 07. Run cells top-to-bottom.

| Section | Topic |
|---|---|
| 1 | Intro and table of contents |
| 2 | Setup: imports and display helpers |
| 3 | Review — from gcd to Bézout: the missing piece |
| 4 | Back-substitution by hand — Python trace |
| 5 | Extended Euclidean algorithm — iterative with full trace |
| 6 | Modular inverse via extended gcd |
| 7 | Affine cipher decryption using computed inverses |
| 8 | RSA private key computation |
| 9 | Exercises |
| 10 | Summary and what comes next |

---
## Section 1 — Introduction

**Module 06** showed that the Euclidean algorithm finds $\gcd(a, b)$ efficiently.
That is enough to **check** whether a modular inverse exists — but it does not give us the inverse itself.

**This module** adds the missing machinery: the **Extended Euclidean Algorithm**, which also finds
integers $s$ and $t$ satisfying

$$\gcd(a, b) = s \cdot a + t \cdot b$$

This is **Bézout's identity**. When $\gcd(a, n) = 1$, it gives $s \equiv a^{-1} \pmod{n}$ directly.

### Why it matters
- **Affine cipher decryption** requires $a^{-1} \pmod{26}$.
- **RSA key generation** requires $e^{-1} \pmod{\phi(n)}$.
- **Elliptic curve cryptography** uses modular inverses at every scalar multiplication.

Python's `pow(a, -1, n)` uses exactly this algorithm internally.

---
## Section 2 — Setup and Display Helpers

In [ ]:
import math

def separator(title=''):
    width = 60
    if title:
        pad = (width - len(title) - 2) // 2
        print('─' * pad + f' {title} ' + '─' * (width - pad - len(title) - 2))
    else:
        print('─' * width)

print('Helpers loaded.')

In [ ]:
def gcd(a, b):
    """Compute gcd(a, b) using the Euclidean algorithm."""
    while b != 0:
        a, b = b, a % b
    return a

# Quick sanity check
test_pairs = [(26, 15), (151, 25), (64, 35), (26, 11)]
print('gcd checks:')
for a, b in test_pairs:
    print(f'  gcd({a}, {b}) = {gcd(a, b)}')

---
## Section 3 — Review: From GCD to Bézout — The Missing Piece

The plain Euclidean algorithm gives us the gcd but **not** the coefficients.
Here we show that by adding a trace, we can see the remainders — but we still
need to walk backward to get $s$ and $t$.

In [ ]:
def gcd_trace(a, b):
    """Run Euclidean algorithm, print every step. Returns gcd."""
    separator(f'gcd({a}, {b}) — forward steps')
    orig_a, orig_b = a, b
    steps = []
    step = 1
    while b != 0:
        q, r = divmod(a, b)
        print(f'  Step {step}: {a} = {q}·{b} + {r}')
        steps.append((a, q, b, r))
        a, b = b, r
        step += 1
    print(f'  → gcd({orig_a}, {orig_b}) = {a}')
    print()
    print('  Notice: the plain algorithm does NOT give us s and t — only the gcd.')
    return a, steps

g, steps_26_15 = gcd_trace(26, 15)
print()
g2, steps_151_25 = gcd_trace(151, 25)

### Exercise 3.1

Run `gcd_trace(64, 35)`. Write down the 4 remainder equations by hand.
Then, starting from the last non-zero remainder equation, try to express 1 in terms of 64 and 35.
You'll need the back-substitution technique from Section 4.

In [ ]:
# Your work here


---
## Section 4 — Back-Substitution by Hand: Python Trace

Back-substitution replaces each intermediate remainder with the expression
from the equation that produced it, working upward through the forward steps.

**Example: find $15^{-1} \pmod{26}$**

Forward steps:
$$26 = 1\cdot15 + 11 \quad 15 = 1\cdot11 + 4 \quad 11 = 2\cdot4 + 3 \quad 4 = 1\cdot3 + 1$$

Back-substitution starts at `1 = 4 - 3` and substitutes upward.

In [ ]:
def back_substitution_trace(a_orig, n_orig, steps):
    """
    Given the list of (a, q, b, r) triples from gcd_trace,
    perform back-substitution and print every step.
    Returns (s, t) where s*a_orig + t*n_orig = 1.
    Assumes gcd = 1.
    """
    separator(f'Back-substitution: {a_orig}⁻¹ mod {n_orig}')

    # Last non-zero remainder equation: a = q*b + r → 1 = a - q*b
    # We track coefficients as (coeff_of_a_orig, coeff_of_n_orig)
    # Start: 1 = 1*prev_a + (-q)*prev_b where prev_a,q,prev_b are from last step

    # Work with (coefficient of a_orig, coefficient of n_orig) for each remainder
    # Represent each remainder r as: r = ca * a_orig + cn * n_orig

    # Build a lookup of coefficients for each number that appeared
    # We know a_orig = 1*a_orig + 0*n_orig, n_orig = 0*a_orig + 1*n_orig
    coeffs = {a_orig: (1, 0), n_orig: (0, 1)}

    print(f'  Seed: {a_orig} = 1·{a_orig} + 0·{n_orig}')
    print(f'  Seed: {n_orig} = 0·{a_orig} + 1·{n_orig}')
    print()

    for (a, q, b, r) in steps:
        if r == 0:
            break
        ca_a, cn_a = coeffs[a]
        ca_b, cn_b = coeffs[b]
        # r = a - q*b
        ca_r = ca_a - q * ca_b
        cn_r = cn_a - q * cn_b
        coeffs[r] = (ca_r, cn_r)
        sign_ca = '+' if ca_r >= 0 else ''
        sign_cn = '+' if cn_r >= 0 else ''
        print(f'  {r} = {a} - {q}·{b}  →  {r} = {sign_ca}{ca_r}·{a_orig} {sign_cn}{cn_r}·{n_orig}')

    # The last remainder computed should be gcd=1
    for (a, q, b, r) in reversed(steps):
        if r == 1:
            ca, cn = coeffs[1]
            break
    else:
        ca, cn = coeffs.get(1, (None, None))

    print()
    print(f'  Result: 1 = {ca}·{a_orig} + {cn}·{n_orig}')
    raw_s = ca
    inv = ca % n_orig
    print(f'  Coefficient of {a_orig}: {raw_s}', end='')
    if raw_s < 0:
        print(f'  →  {raw_s} + {n_orig} = {inv}', end='')
    print()
    print(f'  {a_orig}⁻¹ ≡ {inv} (mod {n_orig})')
    return ca, cn

separator('Example: 15⁻¹ mod 26')
g, steps = gcd_trace(26, 15)
print()
s, t = back_substitution_trace(15, 26, steps)
print()
print(f'Verify: 15 × {s % 26} mod 26 = {(15 * (s % 26)) % 26}')

In [ ]:
# Same for 25⁻¹ mod 151 (one-step case)
separator('Example: 25⁻¹ mod 151')
g, steps = gcd_trace(151, 25)
print()
s, t = back_substitution_trace(25, 151, steps)
print()
print(f'Verify: 25 × {s % 151} mod 151 = {(25 * (s % 151)) % 151}')

### Exercise 4.1

Use `gcd_trace` and `back_substitution_trace` to find $35^{-1} \pmod{64}$.
Verify your answer by computing $35 \times \text{inv} \pmod{64}$.

In [ ]:
# Your work here


---
## Section 5 — Extended Euclidean Algorithm: Iterative Implementation

Back-substitution is conceptually clean but awkward to implement — it requires storing all steps then
working backward. The **iterative** extended Euclidean algorithm tracks coefficients as it goes forward,
producing the same result in a single pass.

At each step we maintain: $a = s_0 \cdot a_0 + t_0 \cdot b_0$ and $b = s_1 \cdot a_0 + t_1 \cdot b_0$.

In [ ]:
def extended_gcd(a, b):
    """Returns (g, s, t) such that g = gcd(a, b) = s*a + t*b."""
    old_r, r = a, b
    old_s, s = 1, 0
    old_t, t = 0, 1
    while r != 0:
        q = old_r // r
        old_r, r = r, old_r - q * r
        old_s, s = s, old_s - q * s
        old_t, t = t, old_t - q * t
    return old_r, old_s, old_t

# Verify on tutorial examples
separator('extended_gcd — tutorial examples')
examples = [
    (15, 26),   # 1 = 7·15 − 4·26
    (25, 151),  # 1 = −6·25 + 1·151
    (35, 64),   # 1 = 11·35 − 6·64
    (11, 26),   # 1 = −7·11 + 3·26
    (7,  12),   # 1 = −5·7 + 3·12
]
print(f'  {"(a,b)":>10}  {"g":>4}  {"s":>6}  {"t":>6}  {"s·a+t·b":>10}  {"match":>6}')
print(f'  {"─"*10}  {"─"*4}  {"─"*6}  {"─"*6}  {"─"*10}  {"─"*6}')
for a, b in examples:
    g, s, t = extended_gcd(a, b)
    check = s * a + t * b
    ok = '✓' if check == g else '✗'
    print(f'  ({a:>3},{b:>3})    {g:>4}  {s:>6}  {t:>6}  {check:>10}  {ok:>6}')

In [ ]:
def extended_gcd_trace(a, b):
    """
    Same as extended_gcd but prints every step including the running
    Bézout coefficients. Returns (g, s, t).
    """
    separator(f'extended_gcd_trace({a}, {b})')
    orig_a, orig_b = a, b
    old_r, r = a, b
    old_s, s = 1, 0
    old_t, t = 0, 1
    step = 1
    print(f'  {"Step":>5}  {"a":>8}  {"b":>8}  {"q":>5}  {"new_r":>8}  {"s":>6}  {"t":>6}')
    print(f'  {"─"*5}  {"─"*8}  {"─"*8}  {"─"*5}  {"─"*8}  {"─"*6}  {"─"*6}')
    while r != 0:
        q = old_r // r
        old_r, r = r, old_r - q * r
        old_s, s = s, old_s - q * s
        old_t, t = t, old_t - q * t
        print(f'  {step:>5}  {old_r:>8}  {r:>8}  {q:>5}  {old_r - q*0:>8}  {old_s:>6}  {old_t:>6}')
        step += 1
    print(f'\n  gcd({orig_a}, {orig_b}) = {old_r}')
    print(f'  Bézout: {old_r} = {old_s}·{orig_a} + {old_t}·{orig_b}')
    print(f'  Verify: {old_s}·{orig_a} + {old_t}·{orig_b} = {old_s*orig_a + old_t*orig_b}')
    return old_r, old_s, old_t

extended_gcd_trace(15, 26)
print()
extended_gcd_trace(35, 64)

### Exercise 5.1

Run `extended_gcd_trace(11, 26)` and compare the output with the back-substitution example
in Tutorial 07. Do the Bézout coefficients match?

Then run `extended_gcd_trace(17, 3120)`. This is the classic small RSA example.
What is $17^{-1} \pmod{3120}$?

In [ ]:
# Your work here


---
## Section 6 — Modular Inverse via Extended GCD

When $\gcd(a, n) = 1$, the Bézout coefficient $s$ satisfies $s \cdot a \equiv 1 \pmod{n}$.
We reduce $s \pmod{n}$ to get the standard representative in $[0, n)$.

In [ ]:
def mod_inverse(a, n):
    """Find a^{-1} mod n using extended_gcd. Raises ValueError if gcd != 1."""
    g, s, t = extended_gcd(a, n)
    if g != 1:
        raise ValueError(f'gcd({a}, {n}) = {g} ≠ 1 — inverse does not exist')
    return s % n

# Tutorial examples
separator('mod_inverse — tutorial examples')
inv_examples = [
    (15,  26,  7),
    (25,  151, 145),
    (35,  64,  11),
    (11,  26,  19),
    (7,   12,  7),
    (21,  26,  5),
    (17,  3120, 2753),
]
print(f'  {"a":>6}  {"n":>6}  {"inv":>6}  {"expected":>8}  {"match":>6}  {"verify":>8}')
print(f'  {"─"*6}  {"─"*6}  {"─"*6}  {"─"*8}  {"─"*6}  {"─"*8}')
for a, n, expected in inv_examples:
    inv = mod_inverse(a, n)
    verify = (a * inv) % n
    ok = '✓' if inv == expected else '✗'
    print(f'  {a:>6}  {n:>6}  {inv:>6}  {expected:>8}  {ok:>6}  {verify:>8}')

In [ ]:
# Compare with Python's built-in pow(a, -1, n)
separator('Comparison with pow(a, -1, n)')
for a, n, _ in inv_examples:
    ours = mod_inverse(a, n)
    builtin = pow(a, -1, n)
    match = '✓' if ours == builtin else '✗'
    print(f'  mod_inverse({a:>4}, {n:>5}) = {ours:>5}   pow({a:>4}, -1, {n:>5}) = {builtin:>5}   {match}')

In [ ]:
# Show that mod_inverse raises correctly when gcd != 1
separator('Non-invertible cases')
bad_pairs = [(13, 26), (6, 26), (299, 391)]
for a, n in bad_pairs:
    try:
        inv = mod_inverse(a, n)
        print(f'  mod_inverse({a}, {n}) = {inv}  ← UNEXPECTED')
    except ValueError as e:
        print(f'  ValueError: {e}')

### Exercise 6.1

1. Find all invertible elements modulo 26 (those with $\gcd(a, 26) = 1$) and list their inverses.
2. What do you notice about the pairs — is each element paired with a different element,
   or can an element be its own inverse?

In [ ]:
# Your work here


---
## Section 7 — Affine Cipher Decryption Using Computed Inverses

The affine cipher encrypts as $C \equiv aP + b \pmod{26}$ and decrypts as:

$$P \equiv a^{-1}(C - b) \pmod{26}$$

We use `mod_inverse` to compute $a^{-1}$, then apply the decryption formula.

In [ ]:
def affine_decrypt(C, a, b):
    """Decrypt ciphertext number C under affine key (a, b) mod 26."""
    a_inv = mod_inverse(a, 26)
    return (a_inv * (C - b)) % 26

def affine_decrypt_string(ciphertext, a, b):
    """Decrypt a string of uppercase letters under affine key (a, b) mod 26."""
    a_inv = mod_inverse(a, 26)
    result = []
    for ch in ciphertext.upper():
        if ch.isalpha():
            C = ord(ch) - ord('A')
            P = (a_inv * (C - b)) % 26
            result.append(chr(P + ord('A')))
        else:
            result.append(ch)
    return ''.join(result)

separator('Affine cipher decryption — tutorial examples')

# Tutorial examples
print('  Example 1: C=13, key=(15,8)  →  P =', affine_decrypt(13, 15, 8))
print('  Check: 15·9+8 = 143 ≡', (15*9+8)%26, '(mod 26)  (should equal 13)')
print()
print('  Example 2: C=20, key=(11,2)  →  P =', affine_decrypt(20, 11, 2))
print('  Check: 11·4+2 = 46 ≡', (11*4+2)%26, '(mod 26)  (should equal 20)')

In [ ]:
# Demonstrate full string decryption
separator('Full string affine decryption')

# Encrypt 'HELLO' with key (7, 3) first, then decrypt
def affine_encrypt_string(plaintext, a, b):
    result = []
    for ch in plaintext.upper():
        if ch.isalpha():
            P = ord(ch) - ord('A')
            C = (a * P + b) % 26
            result.append(chr(C + ord('A')))
        else:
            result.append(ch)
    return ''.join(result)

a, b = 7, 3
plaintext = 'MATHEMATICS'
ciphertext = affine_encrypt_string(plaintext, a, b)
decrypted = affine_decrypt_string(ciphertext, a, b)

print(f'  Key: a={a}, b={b},  a⁻¹ mod 26 = {mod_inverse(a, 26)}')
print(f'  Plaintext:  {plaintext}')
print(f'  Encrypted:  {ciphertext}')
print(f'  Decrypted:  {decrypted}')
print(f'  Round-trip: {"✓" if plaintext == decrypted else "✗"}')

### Exercise 7.1

1. Encrypt the word `CRYPTOGRAPHY` with key $(a=11, b=7)$.
2. Decrypt your result back using `mod_inverse`.
3. What happens if you try key $(a=13, b=5)$? Why does `mod_inverse` raise an error?

In [ ]:
# Your work here


---
## Section 8 — RSA Private Key Computation

RSA key generation requires computing $d = e^{-1} \pmod{\phi(n)}$.
This is exactly what `mod_inverse` does. Here we extend the Module 06 toy RSA example
to show where the extended Euclidean algorithm fits.

In [ ]:
separator('Toy RSA — p=61, q=53, e=17')

p, q = 61, 53
n = p * q
phi_n = (p - 1) * (q - 1)
e = 17

print(f'  p = {p},  q = {q}')
print(f'  n = p × q = {n}')
print(f'  φ(n) = (p−1)(q−1) = {p-1} × {q-1} = {phi_n}')
print(f'  Public exponent e = {e}')
print(f'  gcd(e, φ(n)) = {math.gcd(e, phi_n)}')
print()

# Compute d = e⁻¹ mod φ(n) using extended gcd
g, s, t = extended_gcd(e, phi_n)
d = s % phi_n

print(f'  Extended gcd gives: 1 = {s}·{e} + {t}·{phi_n}')
print(f'  Coefficient of e: s = {s}')
if s < 0:
    print(f'  Adjust: {s} + {phi_n} = {d}')
print(f'  Private exponent d = e⁻¹ mod φ(n) = {d}')
print(f'  Verify: e × d mod φ(n) = {e} × {d} mod {phi_n} = {(e * d) % phi_n}')

In [ ]:
# Full RSA round-trip
separator('RSA encrypt / decrypt round-trip')

M = 42
C = pow(M, e, n)
M2 = pow(C, d, n)

print(f'  Public key:   (e={e}, n={n})')
print(f'  Private key:  (d={d}, n={n})')
print()
print(f'  Plaintext  M = {M}')
print(f'  Encrypted  C = M^e mod n = {M}^{e} mod {n} = {C}')
print(f'  Decrypted  M = C^d mod n = {C}^{d} mod {n} = {M2}')
print(f'  Round-trip: {"✓" if M == M2 else "✗"}')

In [ ]:
# Show the extended gcd trace for the key derivation
separator('Key derivation trace: e⁻¹ mod φ(n)')
extended_gcd_trace(e, phi_n)

### Exercise 8.1

For the toy RSA with $p=61, q=53$:
1. Use `extended_gcd_trace(3, phi_n)` to check whether $e=3$ is a valid public exponent.
2. Try $e=2761$ (a random valid exponent). Compute $d$ and verify the round-trip with $M=99$.
3. Why must we use the **extended** Euclidean algorithm here, rather than the plain one?

In [ ]:
# Your work here — phi_n is already defined above


---
## Section 9 — Exercises

### Exercise 9.1 — Inverse table for a prime modulus

For a prime $p$, every element in $\{1, 2, \ldots, p-1\}$ has a modular inverse.
Build a complete inverse table for $p = 7$ and $p = 11$.
Check whether any element is its own inverse (i.e., $a^{-1} \equiv a \pmod{p}$).

In [ ]:
# Your work here


### Exercise 9.2 — Bézout verification

For each pair below, use `extended_gcd` to find $(g, s, t)$ and verify that $s \cdot a + t \cdot b = g$:

| $a$ | $b$ |
|---|---|
| 252 | 105 |
| 391 | 299 |
| 3120 | 17 |
| 65537 | 3120 |

For the pairs where $g = 1$, also compute the modular inverse and verify it.

In [ ]:
# Your work here


### Exercise 9.3 — Crack the affine cipher

You intercept the ciphertext `MPQKZ`. You know the plaintext letter `A` encrypts to `M`
and the plaintext letter `B` encrypts to `P`.

1. Set up two equations and solve for $a$ and $b$ (hint: $a \cdot 0 + b \equiv 12 \pmod{26}$ gives $b$;
   then $a \cdot 1 + b \equiv 15 \pmod{26}$ gives $a$).
2. Use `mod_inverse` to find $a^{-1}$.
3. Decrypt the full ciphertext.

In [ ]:
# Your work here


---
## Section 10 — Summary and What Comes Next

### Functions built in this notebook

| Function | What it does |
|---|---|
| `extended_gcd(a, b)` | Returns $(g, s, t)$ with $g = \gcd(a,b) = s\cdot a + t\cdot b$ |
| `extended_gcd_trace(a, b)` | Same but prints every step and running coefficients |
| `mod_inverse(a, n)` | Returns $a^{-1} \pmod n$; raises `ValueError` if $\gcd(a,n)\neq 1$ |
| `back_substitution_trace(a, n, steps)` | Shows back-substitution step by step |
| `affine_decrypt(C, a, b)` | Decrypts one ciphertext number |
| `affine_decrypt_string(text, a, b)` | Decrypts a full string |

### Key results confirmed

- $15^{-1} \equiv 7 \pmod{26}$, $\;25^{-1} \equiv 145 \pmod{151}$, $\;35^{-1} \equiv 11 \pmod{64}$
- $11^{-1} \equiv 19 \pmod{26}$, $\;7^{-1} \equiv 7 \pmod{12}$
- $17^{-1} \equiv 2753 \pmod{3120}$ — the classic small RSA private key
- RSA toy example: $p=61, q=53, e=17, d=2753$ — round-trip verified
- `mod_inverse(a, n)` matches Python's `pow(a, -1, n)` for all tested values

---
## What Comes Next

**Module 08 — Bytes as Polynomials** moves into the Galois field $\text{GF}(2^8)$.
Instead of integers modulo $n$, we work with polynomials modulo an irreducible polynomial over $\mathbb{F}_2$.
The extended Euclidean algorithm reappears — applied to polynomials — to compute multiplicative inverses
in $\text{GF}(2^8)$, which is the heart of the AES S-box.